In [1]:
# detection on camera

from pathlib import Path
from ultralytics import YOLO
import cv2

mudra_names = [
    'Alapadmam', 'Anjali', 'Aralam', 'Ardhachandran', 'Ardhapathaka', 'Berunda', 'Bramaram', 'Chakra', 'Chandrakala', 'Chaturam', 'Garuda',
    'Hamsapaksha', 'Hamsasyam', 'Kangulam', 'Kapith', 'Kapotham', 'Karkatta', 'Kartariswastika', 'Katakamukha_1', 'Katakamukha_2',
    'Katakamukha_3', 'Katakavardhana', 'Katrimukha', 'Khatva', 'Kilaka', 'Kurma', 'Matsya', 'Mayura', 'Mrigasirsha', 'Mukulam', 'Mushti',
    'Nagabandha', 'Padmakosha', 'Pasha', 'Pathaka', 'Pushpaputa', 'Sakata', 'Samputa', 'Sarpasirsha', 'Shanka', 'Shivalinga',
    'Shukatundam', 'Sikharam', 'Simhamukham', 'Suchi', 'Swastikam', 'Tamarachudam', 'Tripathaka', 'Trishulam', 'Varaha'
]

trained_model_path = r"E:\sih\mudra_classification_project\yolov8_mudra_cls6\weights\best.pt"
model = YOLO(trained_model_path)

def camera_mudra_detection(threshold=0.5):
    cap = cv2.VideoCapture(0)  # Default webcam
    if not cap.isOpened():
        print("[ERROR] Cannot open camera")
        return

    print("[INFO] Starting camera mudra detection. Press 'q' to quit.")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("[WARN] Can't receive frame. Exiting...")
            break

        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = model.predict(img_rgb, save=False)

        if results and len(results) > 0:
            res = results[0]
            pred_conf = res.probs.top1conf.item()
            if pred_conf > threshold:
                pred_idx = res.probs.top1
                pred_label = mudra_names[pred_idx]
                text = f"{pred_label} ({pred_conf:.2f})"
                cv2.putText(frame, text, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            else:
                # Optionally, display 'Uncertain' or clear text if confidence too low
                cv2.putText(frame, "Uncertain", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        cv2.imshow("Mudra Classification - Press 'q' to exit", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    camera_mudra_detection()


[INFO] Starting camera mudra detection. Press 'q' to quit.

0: 224x224 Kapith 0.53, Katakamukha_2 0.15, Sarpasirsha 0.06, Tamarachudam 0.04, Pathaka 0.02, 33.1ms
Speed: 80.6ms preprocess, 33.1ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)

0: 224x224 Kapith 0.55, Katakamukha_2 0.15, Tamarachudam 0.05, Sarpasirsha 0.04, Pasha 0.03, 24.3ms
Speed: 3.3ms preprocess, 24.3ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)

0: 224x224 Kapith 0.58, Katakamukha_2 0.16, Sarpasirsha 0.04, Tamarachudam 0.03, Pasha 0.02, 23.6ms
Speed: 3.7ms preprocess, 23.6ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)

0: 224x224 Kapith 0.55, Katakamukha_2 0.16, Sarpasirsha 0.04, Tamarachudam 0.04, Pasha 0.03, 31.9ms
Speed: 4.0ms preprocess, 31.9ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)

0: 224x224 Kapith 0.55, Katakamukha_2 0.16, Sarpasirsha 0.04, Tamarachudam 0.04, Pasha 0.02, 27.1ms
Speed: 3.8ms preprocess, 27.1ms inference, 0.

In [3]:
# detection on image
from pathlib import Path
from ultralytics import YOLO
import cv2
import os

# Mudra class names (same list as training)
mudra_names = [
    'Alapadmam', 'Anjali', 'Aralam', 'Ardhachandran', 'Ardhapathaka', 'Berunda', 'Bramaram', 'Chakra', 'Chandrakala', 'Chaturam', 'Garuda',
    'Hamsapaksha', 'Hamsasyam', 'Kangulam', 'Kapith', 'Kapotham', 'Karkatta', 'Kartariswastika', 'Katakamukha_1', 'Katakamukha_2',
    'Katakamukha_3', 'Katakavardhana', 'Katrimukha', 'Khatva', 'Kilaka', 'Kurma', 'Matsya', 'Mayura', 'Mrigasirsha', 'Mukulam', 'Mushti',
    'Nagabandha', 'Padmakosha', 'Pasha', 'Pathaka', 'Pushpaputa', 'Sakata', 'Samputa', 'Sarpasirsha', 'Shanka', 'Shivalinga',
    'Shukatundam', 'Sikharam', 'Simhamukham', 'Suchi', 'Swastikam', 'Tamarachudam', 'Tripathaka', 'Trishulam', 'Varaha'
]

# Path to your trained weights file (update this path as needed)
trained_model_path = r"E:\sih\mudra_classification_project\yolov8_mudra_cls6\weights\best.pt"

# Load your trained model
model = YOLO(trained_model_path)


def predict_mudra(image_path, display=True, save_path=None):
    """
    Predict mudra class for one or more images.
    """
    results = model.predict(image_path, save=False)

    if isinstance(image_path, (str, Path)):
        image_paths = [image_path]
    else:
        image_paths = image_path

    for i, res in enumerate(results):
        pred_idx = res.probs.top1         # index of top-1 class
        pred_label = mudra_names[pred_idx]
        pred_conf = res.probs.top1conf.item()  # confidence of top-1 class


        img_path = str(image_paths[i])
        image = cv2.imread(img_path)
        if image is None:
            print(f"[WARN] Could not read image: {img_path}")
            continue

        cv2.putText(image, f'{pred_label} ({pred_conf:.2f})', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        if save_path:
            os.makedirs(save_path, exist_ok=True)
            out_file = os.path.join(save_path, Path(img_path).name)
            cv2.imwrite(out_file, image)

        if display:
            cv2.imshow("Mudra Classification", image)
            cv2.waitKey(0)
            cv2.destroyAllWindows()

        print(f"[INFO] Image: {img_path} | Predicted: {pred_label} | Confidence: {pred_conf:.2f}")


# Example usage - single image prediction:
test_image = r'E:\sih\bharatanatyam_mudra_dataset\train_13.jpg'
predict_mudra(test_image)

# Uncomment and modify for batch prediction from a folder
# test_folder = r'E:\sih\bharatanatyam_mudra_dataset\test_images'
# for img_file in Path(test_folder).glob("*.jpg"):
#     predict_mudra(img_file, save_path=r'E:\sih\predicted_mudra_images')



image 1/1 E:\sih\bharatanatyam_mudra_dataset\train_13.jpg: 224x224 Sikharam 0.99, Mushti 0.00, Chandrakala 0.00, Ardhachandran 0.00, Tamarachudam 0.00, 15.2ms
Speed: 3.0ms preprocess, 15.2ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)
[INFO] Image: E:\sih\bharatanatyam_mudra_dataset\train_13.jpg | Predicted: Sikharam | Confidence: 0.99
